In [3]:
import requests
import pandas as pd

# -----------------------------
# CONFIG – fill these in
# -----------------------------
API_KEY = "3b339a99725dc0efa820ed87b0db4541"
TOKEN   = "ATTAdc78f81723bc85deb590dd40f9608ec7ec7dda4df5f9396a606240b8e4d4ea20436E01D9"

BOARD_ID = "6879845badb2291c02fc3854"

# Put the list IDs you care about here
LIST_IDS = [
    "698d4586723fd10aac6f01d0"  # Purchase T-1
]
    
OUTPUT_CSV = r"C:\\Users\\User\\OneDrive\\Momentum\\Scripts\\Signal_Archive\\todays_purchase.csv"

# -----------------------------
# Helper: Trello GET wrapper
# -----------------------------
BASE_URL = "https://api.trello.com/1"

def trello_get(path, params=None):
    if params is None:
        params = {}
    auth_params = {"key": API_KEY, "token": TOKEN}
    auth_params.update(params)
    url = f"{BASE_URL}{path}"
    resp = requests.get(url, params=auth_params, timeout=60)
    resp.raise_for_status()
    return resp.json()

# -----------------------------
# Helper: parse description lines (variable_name : value)
# -----------------------------
def parse_description(desc_text):
    result = {}
    if not desc_text:
        return result

    for line in desc_text.splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        var, val = line.split(":", 1)
        result[var.strip()] = val.strip()
    return result

# -----------------------------
# Helper: get custom field value (supports text/number/checkbox/date/list)
# -----------------------------
def get_custom_field_value(cf_item, cf_def):
    """
    cf_item: one item from card["customFieldItems"]
    cf_def:  matching definition from /boards/{BOARD_ID}/customFields
    """
    value = cf_item.get("value") or {}
    cf_type = cf_def.get("type")

    if cf_type == "text":
        return value.get("text", "")

    if cf_type == "number":
        # Trello returns numbers as strings; keep as numeric where possible
        v = value.get("number", "")
        try:
            return float(v) if v != "" else ""
        except ValueError:
            return v

    if cf_type == "checkbox":
        # "true"/"false" strings
        v = value.get("checked", "")
        if v == "true":
            return True
        if v == "false":
            return False
        return ""

    if cf_type == "date":
        return value.get("date", "")

    if cf_type == "list":
        # Dropdown: item stores idValue, definition stores options with text
        id_value = cf_item.get("idValue")
        if not id_value:
            return ""
        options = cf_def.get("options", []) or []
        for opt in options:
            if opt.get("id") == id_value:
                return (opt.get("value") or {}).get("text", "")
        # fallback if not found
        return id_value

    # Unknown type fallback
    return value or ""

# -----------------------------
# 1. Get custom field definitions for the board
# -----------------------------
custom_fields_raw = trello_get(f"/boards/{BOARD_ID}/customFields")

custom_field_defs = {}
custom_field_names = []

for cf in custom_fields_raw:
    cf_id = cf["id"]
    custom_field_defs[cf_id] = cf
    custom_field_names.append(cf["name"])

# -----------------------------
# 2. Get list names for the entire board
# -----------------------------
board_lists = trello_get(f"/boards/{BOARD_ID}/lists", params={"fields": "name"})
list_name_lookup = {lst["id"]: lst["name"] for lst in board_lists}
# -----------------------------
# 3. Collect card data from lists
# -----------------------------
all_rows = []

for list_id in LIST_IDS:
    cards = trello_get(
        f"/lists/{list_id}/cards",
        params={"fields": "name,desc,idList", "customFieldItems": "true"},
    )

    for card in cards:
        row = {}
        row["Title"] = card.get("name", "")
        row["Card_ID"] = card.get("id", "")
        row["List_Name"] = list_name_lookup.get(card.get("idList"), "")

        # Parse description vars
        desc_vars = parse_description(card.get("desc", ""))
        row.update(desc_vars)

        # Add custom field values
        for cf_item in card.get("customFieldItems", []):
            cf_id = cf_item.get("idCustomField")
            cf_def = custom_field_defs.get(cf_id)
            if not cf_def:
                continue
            row[cf_def["name"]] = get_custom_field_value(cf_item, cf_def)

        all_rows.append(row)

# -----------------------------
# 4. Build DataFrame
# -----------------------------
df = pd.DataFrame(all_rows)

# Safe column ordering (won't crash if empty)
base_cols = [c for c in ["Title", "List_Name"] if c in df.columns]
other_cols = [c for c in df.columns if c not in base_cols]

cf_cols_in_df = [name for name in custom_field_names if name in other_cols]
desc_cols = sorted([c for c in other_cols if c not in cf_cols_in_df])

final_cols = base_cols + cf_cols_in_df + desc_cols
df = df.reindex(columns=final_cols)
df['n_components'] = df['PARAM n-value']
df['poly_degree'] = df['PARAM regression']
df['std_multiplier'] = df['PARAM sensitivity']
df['train_window'] = df['PARAM training period']


# -----------------------------
# 5. Save CSV
# -----------------------------
df.to_csv(OUTPUT_CSV, index=False)
print(f"Exported {len(df)} cards to {OUTPUT_CSV}")


Exported 32 cards to C:\\Users\\User\\OneDrive\\Momentum\\Scripts\\Signal_Archive\\todays_purchase.csv


In [4]:
# --- Cell 1: Imports ---
import pandas as pd
import numpy as np
from pandas.tseries.offsets import BDay

In [5]:
import datetime
import requests # Ensure requests is imported
import time

# Ensure TRELLO_API_BASE_URL is correctly set from the kernel state
TRELLO_API_BASE_URL = "https://api.trello.com/1/"

# Assuming TRELLO_KEY and TRELLO_TOKEN are already available from the kernel state
TRELLO_KEY = '3b339a99725dc0efa820ed87b0db4541' # If keys were in secrets, this is how you'd get them.
TRELLO_TOKEN = 'ATTAdc78f81723bc85deb590dd40f9608ec7ec7dda4df5f9396a606240b8e4d4ea20436E01D9'

RATE_LIMIT_SLEEP_SECS = 0.2

def trello_api_request(endpoint, method='GET', params=None, data=None, files=None):
    if params is None:
        params = {}
    if data is None:
        data = {}

    params['key'] = TRELLO_KEY
    params['token'] = TRELLO_TOKEN

    url = f"{TRELLO_API_BASE_URL}{endpoint}"
    try:
        if method == 'GET':
            response = requests.get(url, params=params)
        elif method == 'POST':
            # For POST requests, parameters like name, desc, idList go into 'data'
            # and key/token typically go into 'params' or sometimes also 'data'.
            # Trello API typically uses params for key/token and data for card attributes.
            post_params = {'key': TRELLO_KEY, 'token': TRELLO_TOKEN}
            post_params.update(params)
            response = requests.post(url, params=post_params, data=data, files=files)
        elif method == 'PUT':  # Add PUT method
            put_params = {'key': TRELLO_KEY, 'token': TRELLO_TOKEN}
            response = requests.put(url, params=put_params, data=data)
        else:
            raise ValueError(f"Unsupported HTTP method: {method}")

        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        time.sleep(RATE_LIMIT_SLEEP_SECS)
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"API request failed for endpoint {endpoint} (method: {method}): {e}")
        return None

print("Redefined 'trello_api_request' function to support GET, POST, PUT, and file uploads.")

Redefined 'trello_api_request' function to support GET, POST, PUT, and file uploads.


In [6]:
def attach_plot_to_card(card_id, plot_filepath, filename=None, set_cover=False):
    if not filename:
        filename = Path(plot_filepath).name

    attachments_endpoint = f"cards/{card_id}/attachments"

    try:
        with open(plot_filepath, 'rb') as f:
            files = {'file': (filename, f, 'image/png')}
            attachment_response = trello_api_request(
                attachments_endpoint,
                method='POST',
                data={'name': filename},
                params={'setCover': str(set_cover).lower()},
                files=files
            )

        if attachment_response:
            print(f"Successfully attached {filename} to card {card_id}")
            return attachment_response
        else:
            print(f"Failed to attach {filename} to card {card_id}")
            return None
    except FileNotFoundError:
        print(f"Error: Plot file not found at {plot_filepath}")
        return None
    except Exception as e:
        print(f"An error occurred while attaching plot {filename} to card {card_id}: {e}")
        return None

print("Defined 'attach_plot_to_card' helper function.")

Defined 'attach_plot_to_card' helper function.


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from io import BytesIO
import yfinance as yf
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from IPython.display import display, HTML
from itertools import product
from tqdm import tqdm
from datetime import datetime, timedelta # Import datetime and timedelta
import pytz # Import pytz for timezone handling


# === Configuration ===
TICKER_FILE = r'C:\\Users\\User\\OneDrive\\Momentum\\Scripts\\Signal_Archive\\todays_purchase.csv'
years = 7
initial_capital = 10000
EXCEL_OUTPUT = 'Daily_Reports/Todays_Alerts.xlsx'
PLOTS_DIR = 'Daily_Reports'
buffer_days = 63
risk_factor = 1.0
trade_multiplier = 1.0


os.makedirs(PLOTS_DIR, exist_ok=True)

def get_clean_financial_data(ticker, start_date, end_date):
    data = yf.download(ticker, start=start_date, end=end_date)
    data.columns = data.columns.get_level_values(0)
    data = data.ffill()
    data.index = data.index.tz_localize(None)
    return data

def prepare_data(data):
    data = data.reset_index()
    data['Date_Ordinal'] = pd.to_numeric(data['Date'].map(pd.Timestamp.toordinal))
    scaler = StandardScaler()
    data['Close_Scaled'] = scaler.fit_transform(data[['Close']])
    X = data[['Date_Ordinal']].values
    y = data['Close_Scaled'].values
    return X, y, data, scaler

def train_models_rolling(X, y, n_components, degree, train_window):
    y_pred = np.full_like(y, np.nan)
    for i in range(train_window, len(X)):
        X_train = X[i-train_window:i]
        y_train = y[i-train_window:i]
        gmm = GaussianMixture(n_components=n_components, covariance_type='diag', random_state=42)
        gmm.fit(X_train)
        latent_features = gmm.predict_proba(X_train)
        X_latent = np.hstack([X_train, latent_features])
        poly_reg = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
        poly_reg.fit(X_latent, y_train)
        current_latent = gmm.predict_proba(X[i].reshape(1, -1))
        current_X_latent = np.hstack([X[i].reshape(1, -1), current_latent])
        y_pred[i] = poly_reg.predict(current_X_latent)[0]
    return y_pred

def generate_signals_rolling(data, y_pred, train_window, std_multiplier):
    window = train_window
    residuals = data['Close'] - y_pred
    data['Upper_Bound'] = np.nan
    data['Lower_Bound'] = np.nan
    for i in range(window, len(data)):
        current_std = np.std(residuals.iloc[i-window:i])
        data.loc[data.index[i], 'Upper_Bound'] = y_pred[i] + std_multiplier * current_std
        data.loc[data.index[i], 'Lower_Bound'] = y_pred[i] - std_multiplier * current_std
    data['Buy_Signal'] = np.where(data['Close'] < data['Lower_Bound'], 1, 0)
    data['Sell_Signal'] = np.where(data['Close'] > data['Upper_Bound'], 1, 0)
    return data

def plot_trading_signals(data, ticker):
    """Plot the trading signals with price and bounds"""
    plt.figure(figsize=(14, 8))

    # Plot closing price
    plt.plot(data['Date'], data['Close'], label='Close Price', color='black', alpha=0.8, linewidth=1.5)

    # Plot predicted values
    plt.plot(data['Date'], data['Predicted'], label='Predicted Price', color='blue', alpha=0.7, linestyle='--')

    # Plot bounds
    plt.plot(data['Date'], data['Upper_Bound'], label='Upper Bound', color='red', alpha=0.5, linestyle=':')
    plt.plot(data['Date'], data['Lower_Bound'], label='Lower Bound', color='green', alpha=0.5, linestyle=':')

    # Plot signals
    buy_signals = data[data['Buy_Signal'] == 1]
    sell_signals = data[data['Sell_Signal'] == 1]

    plt.scatter(buy_signals['Date'], buy_signals['Close'],
                label='Buy Signal', color='green', marker='^', s=100, alpha=1)
    plt.scatter(sell_signals['Date'], sell_signals['Close'],
                label='Sell Signal', color='red', marker='v', s=100, alpha=1)

    # Fill between bounds
    plt.fill_between(data['Date'], data['Lower_Bound'], data['Upper_Bound'],
                     color='gray', alpha=0.1, label='Trading Range')

    plt.title(f'{ticker} Trading Signals (Lookahead-Free)')
    plt.xlabel('Date')
    plt.ylabel('Price ($)')
    plt.legend(loc='upper left')
    plt.grid(True)

def plot_recent_trading_signals_with_labels(data, ticker):
    """Plot recent trading signals and label the most recent Predicted, Upper, Lower values"""
    plt.figure(figsize=(14, 8))
    plt.plot(data['Date'], data['Close'], label='Close Price', color='black', alpha=0.8, linewidth=1.5)
    plt.plot(data['Date'], data['Predicted'], label='Predicted Price', color='blue', alpha=0.7, linestyle='--')
    plt.plot(data['Date'], data['Upper_Bound'], label='Upper Bound', color='red', alpha=0.5, linestyle=':')
    plt.plot(data['Date'], data['Lower_Bound'], label='Lower Bound', color='green', alpha=0.5, linestyle=':')

    buy_signals = data[data['Buy_Signal'] == 1]
    sell_signals = data[data['Sell_Signal'] == 1]
    plt.scatter(buy_signals['Date'], buy_signals['Close'], label='Buy Signal', color='green', marker='^', s=100, alpha=1)
    plt.scatter(sell_signals['Date'], sell_signals['Close'], label='Sell Signal', color='red', marker='v', s=100, alpha=1)

    plt.fill_between(data['Date'], data['Lower_Bound'], data['Upper_Bound'], color='gray', alpha=0.1, label='Trading Range')

    # Label latest point
    latest = data.iloc[-1]
    latest_date = latest['Date']
    plt.text(latest_date, latest['Close'], f"Current Price: {latest['Close']:.2f}", color='black', fontsize=10, ha='right', va='bottom')
    plt.text(latest_date, latest['Predicted'], f"Pred: {latest['Predicted']:.2f}", color='blue', fontsize=10, ha='right', va='bottom')
    plt.text(latest_date, latest['Upper_Bound'], f"Upper: {latest['Upper_Bound']:.2f}", color='red', fontsize=10, ha='right', va='bottom')
    plt.text(latest_date, latest['Lower_Bound'], f"Lower: {latest['Lower_Bound']:.2f}", color='green', fontsize=10, ha='right', va='top')

    plt.title(f'{ticker} Trading Signals - Last 6 Months (Annotated)')
    plt.xlabel('Date')
    plt.ylabel('Price ($)')
    plt.legend(loc='upper left')
    plt.grid(True)


def backtest_strategy(data, initial_capital, train_window, buffer_days=buffer_days):
    cash = initial_capital
    shares = 0
    portfolio_value = []
    transactions = []
    position = 'out'
    buy_timer = 0
    sell_timer = 0
    pending_buy_index = None
    pending_sell_index = None
    min_price_since_buy = None
    min_price_date = None
    start_idx = train_window + buffer_days

    for i, row in data.iterrows():
        if i < start_idx:
            portfolio_value.append(cash)
            continue
        current_date = row['Date']
        current_price = row['Close']
        buy_signal = row['Buy_Signal']
        sell_signal = row['Sell_Signal']

        # BUY logic
        if position == 'out':
            if buy_signal:
                buy_timer = 3
                pending_buy_index = i
            elif buy_timer > 0:
                buy_timer -= 1
                if buy_signal:
                    buy_timer = 3
                    pending_buy_index = i

                if buy_timer == 0 and pending_buy_index is not None:
                    execution_index = i
                    if execution_index < len(data):
                        execution_date = data.loc[execution_index, 'Date']
                        execution_price = data.loc[execution_index, 'Close']
                        shares_to_buy = cash // execution_price
                        trade_value = shares_to_buy * execution_price
                        commission = 30 if trade_value < 10000 else trade_value * 0.003
                        cost = trade_value + commission

                        if shares_to_buy > 0:
                            cash -= cost
                            shares += shares_to_buy
                            position = 'long'
                            min_price_since_buy = execution_price
                            min_price_date = execution_date
                            transactions.append({
                                'Date': execution_date,
                                'Type': 'BUY',
                                'Price': execution_price,
                                'Shares': shares_to_buy,
                                'Value': trade_value,
                                'Commission': commission,
                                'Cash': cash,
                                'Position': position,
                                'Days_Held': 0,
                                'Profit_Loss': 0,
                                'Pct_Return': 0,
                                'Annualized_Return_%': 0,
                                'StopLoss%': np.nan,
                                'StopLoss_Date': pd.NaT
                            })

                    pending_buy_index = None

        # SELL logic
        elif position == 'long':
            if current_price < min_price_since_buy:
                min_price_since_buy = current_price
                min_price_date = current_date
            if sell_signal:
                sell_timer = 3
                pending_sell_index = i
            elif sell_timer > 0:
                sell_timer -= 1
                if sell_signal:
                    buy_timer = 3
                    pending_sell_index = i

                if sell_timer == 0 and pending_sell_index is not None:
                    execution_index = i
                    if execution_index < len(data):
                        execution_date = data.loc[execution_index, 'Date']
                        execution_price = data.loc[execution_index, 'Close']
                        trade_value = shares * execution_price
                        commission = 30 if trade_value < 10000 else trade_value * 0.003
                        proceeds = trade_value - commission
                        cash += proceeds

                        buy_price = transactions[-1]['Price']
                        buy_value = transactions[-1]['Value']
                        buy_date = transactions[-1]['Date']
                        days_held = (execution_date - buy_date).days

                        profit_loss = trade_value - buy_value - 2 * commission
                        pct_return = (trade_value - buy_value) / buy_value * 100
                        annualized_return = ((1 + pct_return / 100) ** (365 / days_held) - 1) * 100 if days_held > 0 else np.nan
                        stop_loss_pct = ((min_price_since_buy - buy_price) / buy_price * 100
                                         if min_price_since_buy is not None and min_price_since_buy < buy_price
                                         else 0)

                        transactions.append({
                            'Date': execution_date,
                            'Type': 'SELL',
                            'Price': execution_price,
                            'Shares': shares,
                            'Value': trade_value,
                            'Commission': commission,
                            'Cash': cash,
                            'Position': 'out',
                            'Days_Held': days_held,
                            'Profit_Loss': profit_loss,
                            'Pct_Return': pct_return,
                            'Annualized_Return_%': annualized_return,
                            'StopLoss%': stop_loss_pct,
                            'StopLoss_Date': min_price_date
                        })

                        shares = 0
                        position = 'out'
                        buy_timer = 0
                        sell_timer = 0
                        pending_buy_index = None
                        pending_sell_index = None
                        min_price_since_buy = None
                        min_price_date = None

        current_value = cash + shares * current_price
        portfolio_value.append(current_value)

    portfolio_df = pd.DataFrame({'Date': data['Date'], 'Portfolio_Value': portfolio_value, 'Stock_Price': data['Close']})
    portfolio_df = portfolio_df.iloc[start_idx:].reset_index(drop=True)
    transactions_df = pd.DataFrame(transactions)
    num_trades = len(transactions_df)

    # Handle open position with virtual SELL
    if num_trades % 2 == 1 and transactions_df.iloc[-1]['Type'] == 'BUY':
        last_buy = transactions_df.iloc[-1]
        buy_price = last_buy['Price']
        buy_date = last_buy['Date']
        shares = last_buy['Shares']
        commission = last_buy['Commission']

        current_price = portfolio_df['Stock_Price'].iloc[-1]
        current_date = portfolio_df['Date'].iloc[-1]

        days_held = (current_date - buy_date).days
        profit_loss = (current_price * shares) - (buy_price * shares) - commission
        pct_return = (current_price - buy_price) / buy_price * 100
        annualized_return = ((1 + pct_return / 100) ** (365 / days_held) - 1) * 100 if days_held > 0 else np.nan

        stop_loss_pct = ((min_price_since_buy - buy_price) / buy_price * 100
                         if min_price_since_buy is not None and min_price_since_buy < buy_price
                         else 0)

        virtual_sell = {
            'Date': current_date,
            'Type': 'SELL (Open)',
            'Price': current_price,
            'Shares': shares,
            'Value': current_price * shares,
            'Commission': 0,
            'Cash': transactions_df.iloc[-1]['Cash'],
            'Position': 'Unrealised Gain',
            'Days_Held': days_held,
            'Profit_Loss': profit_loss,
            'Pct_Return': pct_return,
            'Annualized_Return_%': annualized_return,
            'StopLoss%': stop_loss_pct,
            'StopLoss_Date': min_price_date
        }
        transactions_df = pd.concat([transactions_df, pd.DataFrame([virtual_sell])], ignore_index=True)

    return portfolio_df, transactions_df

def calculate_performance_metrics(portfolio_df, transactions_df, initial_capital, train_window, years, buffer_days=buffer_days, r_value=1500):
    """Calculate and display key performance metrics"""
    final_value = portfolio_df['Portfolio_Value'].iloc[-1]
    total_return = (final_value - initial_capital) / initial_capital * 100
    portfolio_df['Peak'] = portfolio_df['Portfolio_Value'].cummax()
    portfolio_df['Drawdown'] = (portfolio_df['Portfolio_Value'] - portfolio_df['Peak']) / portfolio_df['Peak']
    max_drawdown = portfolio_df['Drawdown'].min() * 100
    total_days_held = transactions_df['Days_Held'].dropna().sum()
    total_days = (portfolio_df['Date'].iloc[-1] - portfolio_df['Date'].iloc[0]).days
    pct_days_held = (total_days_held / total_days * 100) if total_days > 0 else 0
    start_price = portfolio_df['Stock_Price'].iloc[0]
    end_price = portfolio_df['Stock_Price'].iloc[-1]
    buy_and_hold_return = (end_price - start_price) / start_price * 100
    buy_and_hold_annualised = buy_and_hold_return / (7 - ((train_window + buffer_days)/252))
    buy_and_hold_size = initial_capital / start_price
    buy_and_hold_value = buy_and_hold_size * portfolio_df['Stock_Price'].iloc[-1]
    available_trading_days = (years - ((train_window + buffer_days)/252)) * 365
    num_trades = len(transactions_df)
    num_winning_trades = 0
    trade_returns = []
    for i in range(0, num_trades-1, 2):
        if i+1 < num_trades:
            buy_price = transactions_df.iloc[i]['Price']
            sell_price = transactions_df.iloc[i+1]['Price']
            trade_return = (sell_price - buy_price) / buy_price * 100
            trade_returns.append(trade_return)
            if trade_return > 0:
                num_winning_trades += 1

    win_rate = num_winning_trades / len(trade_returns) * 100 if len(trade_returns) > 0 else 0
    avg_trade_days = total_days_held / (num_trades/2)
    strat_annualised = total_return / (total_days_held/365) if total_days_held > 0 else np.nan
    return_diff = final_value - buy_and_hold_value
    outperforms_bh = "Yes" if strat_annualised > buy_and_hold_annualised else "No"
    stoploss_vals = transactions_df['StopLoss%'].dropna()
    if not stoploss_vals.empty:
        min_idx = stoploss_vals.idxmin()
        worst_stoploss = stoploss_vals.loc[min_idx]
        worst_stoploss_date = transactions_df.loc[min_idx, 'StopLoss_Date']
    else:
        worst_stoploss = np.nan
        worst_stoploss_date = pd.NaT
    mod_stoploss = worst_stoploss if worst_stoploss < -10 else -10
    dollar_per_day = (final_value - initial_capital)/ total_days_held
    trade_freq = (num_trades - 10)/10 if num_trades < 11 else (num_winning_trades - 10)/20
    score = dollar_per_day * (risk_factor*((100 + worst_stoploss)/100)) * (trade_multiplier * ( 1 + trade_freq))
    run_date = portfolio_df['Date'].iloc[-1]
    stoploss_price = end_price * ((100 + mod_stoploss)/100)
    risk_price = end_price - stoploss_price
    au_size = (r_value * 0.88)/risk_price
    us_size = (r_value * 0.6)/risk_price
    uk_size = (r_value * 0.44)/risk_price
    nz_size = r_value/risk_price
    hk_size = (r_value * 4.54)/risk_price
    si_size = (r_value * 0.77)/risk_price

    print(f"Initial Capital: ${initial_capital:.2f}")
    print("\n=== Buy & Hold Strategy Performance ===")
    print(f"Buy & Hold Value: ${buy_and_hold_value:.2f}")
    print(f"Buy & Hold Return: {buy_and_hold_return:.2f}%")
    print(f"Total Days Held: {available_trading_days:.2f}")
    print("\n=== Gaussian Strategy Performance ===")
    print(f"Portfolio Value: ${final_value:.2f}")
    print(f"Difference to Buy & Hold: ${return_diff:.2f}")
    print(f"Total Return (Realised & Unrealised): {total_return:.2f}%")
    print(f"Number of Transactions: {num_trades:.2f}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Total Days Held: {total_days_held}")
    print(f"Worst Drawdown: {worst_stoploss:.2f}%")
    print(f"Worst drawdown Date: {worst_stoploss_date}")
    print(f"% Days Held over Strategy Period: {pct_days_held:.2f}%")
    print("\n=== Annualised Gaussian Strategy Performance ===")
    print(f"Buy & Hold Annualised rate: {buy_and_hold_annualised:.2f}%")
    print(f"Gaussian Annualised Rate: {strat_annualised:.2f}%")
    print(f" $ per day: ${dollar_per_day:.2f}")
    print("\n=== Conclusion ===")
    print(f"Strategy Outperforms Buy & Hold?: {outperforms_bh}")
    print(f"Score : {score:.2f}")
    print("\n\n=== Position sizing ===")
    print(f"Stoploss Price: ${stoploss_price:.2f}   with a Risk Premium of ${risk_price:.2f}")
    print(f" Australia quantity: {au_size:.2f}, USA quantity: {us_size:.2f}, London quantity: {uk_size:.2f}")
    print(f" New Zealand quantity: {nz_size:.2f}, Hong Kong quantity: {hk_size:.2f}, Singapore quantity: {si_size:.2f}")

def save_plot_to_file(plot_func, *args, filename, **kwargs):
    # Removed plt.figure() as plotting functions create their own figures
    plot_func(*args, **kwargs)
    plt.tight_layout()
    plt.savefig(filename, format='png', bbox_inches='tight')
    plt.close('all')
    return filename

def plot_bh_comparison(portfolio_df, ticker, train_window):
    plt.plot(portfolio_df['Date'].iloc[train_window:], portfolio_df['Portfolio_Value'].iloc[train_window:], label='Strategy', color='blue')
    start_price = portfolio_df['Stock_Price'].iloc[train_window]
    bh_line = portfolio_df['Stock_Price'] * initial_capital / start_price
    plt.plot(portfolio_df['Date'].iloc[train_window:], bh_line.iloc[train_window:], label='Buy & Hold', color='red', alpha=0.7)
    plt.title(f'Strategy vs Buy & Hold: {ticker}')
    plt.legend()
    plt.grid(True)

def plot_adx_6m(data, ticker):
    """Plot ADX (+DI/-DI) for the last 6 months"""
    plt.figure(figsize=(14, 4))
    plt.plot(data['Date'], data['ADX'], label='ADX', color='blue', linewidth=1.5)
    plt.plot(data['Date'], data['PlusDI'], label='+ADX', color='green', linewidth=1.5)
    plt.plot(data['Date'], data['MinusDI'], label='-ADX', color='red', linewidth=1.5)
    plt.axhline(25, linestyle=':', linewidth=1)  # common trend-strength threshold
    # Label latest point
    latest = data.iloc[-1]
    latest_date = latest['Date']
    plt.text(latest_date, latest['MinusDI'], f"Minus DI: {latest['MinusDI']:.2f}", color='black', fontsize=10, ha='right', va='bottom')
    plt.title(f'{ticker} ADX  — Last 6 Months')
    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend(loc='upper left')
    plt.grid(True)

def calculate_adx(df, period: int = 14):
    """
    Wilder's ADX (+DI / -DI and ADX) using High/Low/Close.
    Adds columns: 'PlusDI', 'MinusDI', 'ADX' to df (same length, NaNs at head).
    """
    high = df['High']
    low  = df['Low']
    close = df['Close']

    up_move   = high.diff()
    down_move = (-low.diff())

    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)

    tr1 = high - low
    tr2 = (high - close.shift()).abs()
    tr3 = (low - close.shift()).abs()
    tr  = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    # Wilder's smoothing via EMA with alpha=1/period
    atr      = tr.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    plus_dmS = pd.Series(plus_dm, index=df.index).ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    minus_dmS= pd.Series(minus_dm, index=df.index).ewm(alpha=1/period, adjust=False, min_periods=period).mean()

    plus_di  = 100 * (plus_dmS / atr)
    minus_di = 100 * (minus_dmS / atr)

    dx = 100 * (plus_di.subtract(minus_di).abs() / (plus_di + minus_di))
    adx = dx.ewm(alpha=1/period, adjust=False, min_periods=period).mean()

    df['PlusDI']  = plus_di
    df['MinusDI'] = minus_di
    df['ADX']     = adx
    return df

def collect_performance_text(portfolio_df, transactions_df, initial_capital, train_window, years):
    import io
    import sys
    old_stdout = sys.stdout
    sys.stdout = mystdout = io.StringIO()
    calculate_performance_metrics(portfolio_df, transactions_df, initial_capital, train_window, years)
    sys.stdout = old_stdout
    return mystdout.getvalue()

def process_ticker_to_excel(ticker, writer, start_date, end_date, n_components, degree, std_multiplier, train_window, Card_ID):
    try:
        data = get_clean_financial_data(ticker, start_date, end_date)
        X, y, data, scaler = prepare_data(data)
        y_pred_scaled = train_models_rolling(X, y, n_components, degree, train_window)
        y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        data['Predicted'] = y_pred
        data = generate_signals_rolling(data, y_pred, train_window, std_multiplier)

        data = calculate_adx(data, period=14)
        portfolio_df, transactions_df = backtest_strategy(data, initial_capital, train_window)


        # === Format and Display Transactions ===
        formatted_log = transactions_df.copy()
        numeric_cols = formatted_log.select_dtypes(include=['float64', 'float32']).columns
        formatted_log[numeric_cols] = formatted_log[numeric_cols].round(2)
        formatted_log = formatted_log.fillna("")
        print("\n=== Transaction Log (Completed Trades) ===")
        display(formatted_log[formatted_log['Type'].isin(['BUY', 'SELL'])])

        # === Display Open Trade, if any  ====
        open_trade_df = transactions_df[transactions_df['Type'] == 'SELL (Open)']

        if not open_trade_df.empty:
            print("\n=== Open Trade ===")
            display(open_trade_df)

        #===Results ===
        calculate_performance_metrics(portfolio_df, transactions_df, initial_capital, train_window, years)


        six_months_ago = data['Date'].max() - pd.DateOffset(months=6)
        recent_data = data[data['Date'] >= six_months_ago].copy()
        p3_file = save_plot_to_file(plot_recent_trading_signals_with_labels, recent_data.copy(), ticker, filename=os.path.join(PLOTS_DIR, f"{ticker}_6_months.png"))
        p4_file = save_plot_to_file(
            plot_adx_6m, recent_data.copy(), ticker,
            filename=os.path.join(PLOTS_DIR, f"{ticker}_6_months_ADX.png")
        )
        p1_file = save_plot_to_file(plot_trading_signals, data.copy(), ticker, filename=os.path.join(PLOTS_DIR, f"{ticker}_signal_chart.png"))

        return p1_file, p3_file, p4_file

    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        return None, None, None

def main():
    tickers_df = pd.read_csv(TICKER_FILE)
    start_date = pd.Timestamp.today() - pd.DateOffset(years=years)
    end_date = pd.Timestamp.today()

    # Define New Zealand timezone
    nz_timezone = pytz.timezone('Pacific/Auckland')

    with pd.ExcelWriter(EXCEL_OUTPUT, engine='xlsxwriter') as writer:
        for idx, row in tickers_df.iterrows():
            ticker = row['Ticker']
            try:
                n_components = int(row['n_components'])
                degree = int(row['poly_degree'])
                std_multiplier = float(row['std_multiplier'])
                train_window = int(row['train_window'])
                card_id = row['Card_ID'] # Changed to lowercase card_id for consistency
                score = row['Score'] # Get Score for comment

                print(f"\u2007\U0001f50d Processing {ticker} with n_components={n_components}, degree={degree}, std_multiplier={std_multiplier}, train_window={train_window}")
                p1_file, p3_file, p4_file = process_ticker_to_excel(ticker, writer, start_date, end_date, n_components, degree, std_multiplier, train_window, card_id)

                if card_id and pd.notna(card_id):
                    # 1. Set the card's due date to 2 business days in the future at 9am (NZT)
                    now_nzt = datetime.now(nz_timezone)
                    future_datetime_nzt = (pd.Timestamp(now_nzt).tz_localize(None) + pd.tseries.offsets.BDay(2)).to_pydatetime()
                    future_datetime_nzt = future_datetime_nzt.replace(hour=9, minute=0, second=0, microsecond=0, tzinfo=nz_timezone)
                    trello_due_date = future_datetime_nzt.isoformat()

                    update_due_date_endpoint = f"cards/{card_id}"
                    update_due_date_payload = {'due': trello_due_date}
                    due_date_response = trello_api_request(update_due_date_endpoint, method='PUT', data=update_due_date_payload)

                    if due_date_response:
                        print(f"Successfully set due date for card {card_id} to {trello_due_date}")
                    else:
                        print(f"Failed to set due date for card {card_id}")

                    # 2. Add a comment to the card
                    comment_text = (f"Purchase parameter settings are ({n_components}, {degree}, {std_multiplier}, "
                                    f"{train_window}) with a HPI of {score:.2f}.")

                    add_comment_endpoint = f"cards/{card_id}/actions/comments"
                    add_comment_payload = {'text': comment_text}
                    comment_response = trello_api_request(add_comment_endpoint, method='POST', data=add_comment_payload)

                    if comment_response:
                        print(f"Successfully added comment to card {card_id}")
                    else:
                        print(f"Failed to add comment to card {card_id}")

                    # Attach plots (existing logic)
                    if p3_file:
                        attach_plot_to_card(card_id, p3_file, filename=f'{ticker}_6_months.png')
                    if p4_file:
                        attach_plot_to_card(card_id, p4_file, filename=f'{ticker}_6_months_ADX.png')
                    if p1_file:
                        attach_plot_to_card(card_id, p1_file, filename=f'{ticker}_signal_chart.png', set_cover=True)

            except Exception as e:
                print(f"\u26a0\ufe0f Skipping {ticker} due to missing/invalid parameters: {e}")

if __name__ == '__main__':
    main()

 🔍 Processing 0590.HK with n_components=2, degree=3, std_multiplier=3.0, train_window=189
YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed



=== Transaction Log (Completed Trades) ===


,Date,Type,Price,Shares,Value,Commission,Cash,Position,Days_Held,Profit_Loss,Pct_Return,Annualized_Return_%,StopLoss%,StopLoss_Date
0,2025-08-06,BUY,20.11,497.0,9996.33,30.00,-26.33,long,0,0.00,0.00,0.0,,NaT
1,2025-09-12,SELL,26.67,497.0,13253.43,39.76,13187.34,out,37,3177.58,32.58,1515.56,0.0,2025-08-06
2,2025-10-02,BUY,24.16,545.0,13167.59,39.50,-19.76,long,0,0.00,0.00,0.0,,NaT
3,2026-02-02,SELL,31.14,545.0,16971.30,50.91,16900.63,out,123,3701.88,28.89,112.35,-6.81,2025-11-04
4,2026-03-13,BUY,26.54,636.0,16879.44,50.64,-29.45,long,0,0.00,0.00,0.0,,NaT



=== Open Trade ===


...


=== Transaction Log (Completed Trades) ===


,Date,Type,Price,Shares,Value,Commission,Cash,Position,Days_Held,Profit_Loss,Pct_Return,Annualized_Return_%,StopLoss%,StopLoss_Date
0,2021-07-30,BUY,4.18,2393.0,9999.91,30.00,-29.91,long,0,0.00,0.00,0.0,,NaT
1,2021-08-26,SELL,4.51,2393.0,10782.68,32.35,10720.42,out,27,718.08,7.83,176.99,-7.44,2021-08-02
2,2021-09-30,BUY,3.97,2697.0,10718.88,32.16,-30.62,long,0,0.00,0.00,0.0,,NaT
3,2021-11-01,SELL,4.50,2697.0,12130.43,36.39,12063.42,out,32,1338.76,13.17,310.03,-12.14,2021-10-15
4,2023-03-24,BUY,2.89,4172.0,12061.47,36.18,-34.24,long,0,0.00,0.00,0.0,,NaT
5,2023-07-28,SELL,3.35,4172.0,13957.21,41.87,13881.10,out,126,1812.00,15.72,52.63,-2.65,2023-03-27
6,2023-08-25,BUY,2.72,5111.0,13878.43,41.64,-38.97,long,0,0.00,0.00,0.0,,NaT
7,2024-10-10,SELL,3.10,5111.0,15823.77,47.47,15737.33,out,412,1850.39,14.02,12.32,-11.44,2024-02-15
8,2025-04-14,BUY,3.28,4793.0,15735.86,47.21,-45.74,long,0,0.00,0.00,0.0,,NaT
9,2026-02-03,SELL,3.57,4793.0,17111.01,51.33,17013.94,out,295,1272.48,8.74,10.92,-2.85,2025-04-16



=== Open Trade ===


,Date,Type,Price,Shares,Value,Commission,Cash,Position,Days_Held,Profit_Loss,Pct_Return,Annualized_Return_%,StopLoss%,StopLoss_Date
11,2026-03-13,SELL (Open),3.87,4396.0,17012.519497,0.0,-49.620925,Unrealised Gain,0,-51.037558,0.0,NaN,0.0,2026-03-13


Initial Capital: $10000.00

=== Buy & Hold Strategy Performance ===
Buy & Hold Value: $8872.59
Buy & Hold Return: -11.27%
Total Days Held: 2281.25

=== Gaussian Strategy Performance ===
Portfolio Value: $16962.90
Difference to Buy & Hold: $8090.31
Total Return (Realised & Unrealised): 69.63%
Number of Transactions: 12.00
Win Rate: 83.33%
Total Days Held: 892
Worst Drawdown: -12.14%
Worst drawdown Date: 2021-10-15 00:00:00
% Days Held over Strategy Period: 39.17%

=== Annualised Gaussian Strategy Performance ===
Buy & Hold Annualised rate: -1.80%
Gaussian Annualised Rate: 28.49%
 $ per day: $7.81

=== Conclusion ===
Strategy Outperforms Buy & Hold?: Yes
Score : 5.14


=== Position sizing ===
Stoploss Price: $3.40   with a Risk Premium of $0.47
 Australia quantity: 2809.62, USA quantity: 1915.65, London quantity: 1404.81
 New Zealand quantity: 3192.75, Hong Kong quantity: 14495.07, Singapore quantity: 2458.41
Successfully set due date for card 69b1e5666aeebfd67696542f to 2026-03-17T09:00

In [9]:
import pandas as pd
import requests
from datetime import datetime

# Trello API credentials
TRELLO_API_KEY   = "3b339a99725dc0efa820ed87b0db4541"
TRELLO_API_TOKEN = "ATTAdc78f81723bc85deb590dd40f9608ec7ec7dda4df5f9396a606240b8e4d4ea20436E01D9"

# Input file
INPUT_CSV = r"C:\\Users\\User\\OneDrive\\Momentum\\Scripts\\Signal_Archive\\todays_purchase.csv"

# Trello API base
TRELLO_BASE_URL = "https://api.trello.com/1"

# Custom field names on your Trello board (must match exactly)
CF_MAP = {
    "PARAM n-value": "n_components",
    "PARAM regression": "poly_degree",
    "PARAM sensitivity": "std_multiplier",
    "PARAM training period": "train_window",
    "HPI": "Score",
}

def trello_params(extra=None):
    p = {"key": TRELLO_API_KEY, "token": TRELLO_API_TOKEN}
    if extra:
        p.update(extra)
    return p

def build_description(row: pd.Series) -> str:
    """
    Builds the Trello card description from a dataframe row.
    """
    # Safely get 'Transactions' value, default to 0 if not found, then convert to numeric
    transactions_val = pd.to_numeric(row.get('Transactions', 0), errors='coerce')
    # Divide by 2 if it's a valid number, otherwise default to empty string
    num_trades_str = f"{transactions_val / 2:.0f}" if pd.notna(transactions_val) else ''

    lines = [
       f"Ticker: {row.get('Ticker','')}",
        "",
        f"Key Parameters: ",
        f"n-value: {row.get('n_components','')}",
        f"Regression: {row.get('poly_degree','')}",
        f"Sensitivity: {row.get('std_multiplier','')}",
        f"Training Period: {row.get('train_window','')}",
        "",
        f"Number of trades: {row.get('Number of trades','')}",
        f"Last 12 months: {row.get('Last 12 months','')}",
        f"Score: {row.get('Score','')}",
        "",
        f"Maximum Stop Loss %: {row.get('Maximum Stop Loss %','')}",
        "",
        f"Industry: {row.get('Industry','')}",
        f"Sector: {row.get('Sector','')}",
        f"Exchange: {row.get('Exchange','')}",
        "",
        f"Updated: {row.get('Run Date','')}",
    ]
    return "\n".join(lines)

def to_number_string(x):
    """
    Trello expects numeric custom field values as strings (e.g., {"number":"4"}).
    Returns None if missing/unparseable.
    """
    if pd.isna(x):
        return None
    try:
        # allow ints, floats, numeric strings
        val = float(str(x).strip())
        # Keep integers clean if they are whole numbers
        if val.is_integer():
            return str(int(val))
        return str(val)
    except Exception:
        return None

def get_card(card_id: str, timeout=15):
    url = f"{TRELLO_BASE_URL}/cards/{card_id}"
    r = requests.get(url, params=trello_params(), timeout=timeout)
    r.raise_for_status()
    return r.json()

def get_board_custom_fields(board_id: str, timeout=15):
    """
    Returns list of board custom fields.
    """
    url = f"{TRELLO_BASE_URL}/boards/{board_id}/customFields"
    r = requests.get(url, params=trello_params(), timeout=timeout)
    r.raise_for_status()
    return r.json()

def update_card_description(card_id: str, desc: str, timeout=15):
    url = f"{TRELLO_BASE_URL}/cards/{card_id}"
    r = requests.put(url, params=trello_params({"desc": desc}), timeout=timeout)
    r.raise_for_status()
    return r.json()

def update_card_custom_field_number(card_id: str, custom_field_id: str, number_as_string: str, timeout=15):
    """
    PUT /cards/{idCard}/customField/{idCustomField}/item
    Body: {"value":{"number":"<string>"}}
    """
    url = f"{TRELLO_BASE_URL}/cards/{card_id}/customField/{custom_field_id}/item"
    payload = {"value": {"number": number_as_string}}
    r = requests.put(url, params=trello_params(), json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()

df = pd.read_csv(INPUT_CSV)

# Required columns for this automation
required_cols = [
    "Card_ID", "Ticker",
    "n_components", "poly_degree", "std_multiplier", "train_window",
    "Number of trades", "Score", "Last 12 months",
    "Maximum Stop Loss %", "Run Date" , "Sector", "Industry", "Exchange"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in {INPUT_CSV}: {missing}")

# Normalize key columns
df["Card_ID"] = df["Card_ID"].astype(str).str.strip()
df["Ticker"] = df["Ticker"].astype(str).str.strip()

ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
success_rows = []
error_rows = []

# Cache board custom-field ID lookups (board_id -> name->field_obj)
board_cf_cache = {}

for i, row in df.iterrows():
    card_id = row["Card_ID"]

    if not card_id or card_id.lower() == "nan":
        error_rows.append({"Row": i, "Card_ID": card_id, "Ticker": row.get("Ticker",""), "Step": "input", "Error": "Missing Card_ID", "Datestamp": ts})
        continue

    try:
        # 1) Get card (to discover board)
        card = get_card(card_id)
        board_id = card.get("idBoard")
        if not board_id:
            raise RuntimeError("Could not determine idBoard from card.")

        # 2) Load board custom fields (cached)
        if board_id not in board_cf_cache:
            cfs = get_board_custom_fields(board_id)
            # map by exact name
            board_cf_cache[board_id] = {cf.get("name"): cf for cf in cfs}

        cf_by_name = board_cf_cache[board_id]

        # 3) Update description
        desc = build_description(row)
        update_card_description(card_id, desc)

        # 4) Update custom fields
        for cf_name, source_col in CF_MAP.items():
            cf_obj = cf_by_name.get(cf_name)
            if not cf_obj:
                raise RuntimeError(f"Custom field '{cf_name}' not found on board {board_id}. Check spelling/case.")

            cf_id = cf_obj.get("id")
            cf_type = cf_obj.get("type")

            if cf_type != "number":
                raise RuntimeError(f"Custom field '{cf_name}' is type '{cf_type}', expected 'number'.")

            num_str = to_number_string(row.get(source_col))
            if num_str is None:
                # Decide policy: skip or fail.
                # Here: skip update for that field but record it.
                error_rows.append({
                    "Row": i, "Card_ID": card_id, "Ticker": row.get("Ticker",""),
                    "Step": f"custom_field:{cf_name}", "Error": f"Value in column '{source_col}' is blank/non-numeric; skipped.", "Datestamp": ts
                })
                continue

            update_card_custom_field_number(card_id, cf_id, num_str)

        success_rows.append({"Row": i, "Card_ID": card_id, "Ticker": row.get("Ticker",""), "Datestamp": ts})

    except requests.HTTPError as e:
        # Include response text if available
        resp_text = ""
        try:
            resp_text = e.response.text
        except Exception:
            pass
        error_rows.append({"Row": i, "Card_ID": card_id, "Ticker": row.get("Ticker",""), "Step": "http", "Error": f"{e} {resp_text}".strip(), "Datestamp": ts})
    except Exception as e:
        error_rows.append({"Row": i, "Card_ID": card_id, "Ticker": row.get("Ticker",""), "Step": "exception", "Error": str(e), "Datestamp": ts})

# Write logs
out_success = r"C:\\Users\\User\\OneDrive\\Momentum\\Scripts\\Signal_Archive\\purchase_trello_cards_update_success.csv"
out_errors   = r"C:\\Users\\User\\OneDrive\\Momentum\\Scripts\\Signal_Archive\\purchase_trello_cards_update_errors.csv"

pd.DataFrame(success_rows).to_csv(out_success, index=False)
pd.DataFrame(error_rows).to_csv(out_errors, index=False)

print(f"Done. Success: {len(success_rows)} | Errors/Warnings: {len(error_rows)}")
print(f"Success log: {out_success}")
print(f"Error log:   {out_errors}")

Done. Success: 32 | Errors/Warnings: 0
Success log: C:\\Users\\User\\OneDrive\\Momentum\\Scripts\\Signal_Archive\\purchase_trello_cards_update_success.csv
Error log:   C:\\Users\\User\\OneDrive\\Momentum\\Scripts\\Signal_Archive\\purchase_trello_cards_update_errors.csv
